# Micro-TERRA anchor-ratio ablation

Stage 1 only (no quantization), 500 steps each, on a clean held-out validation set.
Compares `--anchor-every` 8 (7:1), 4 (3:1) and 2 (1:1). Needs a GPU runtime (A100).

In [ ]:
%cd /content
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /content/terra_llm && git clone -q -b clean-val-anchor-ablation https://github.com/sswapnil2/terra-llm.git /content/terra_llm
%cd /content/terra_llm
!pip install -q --upgrade "bitsandbytes>=0.43.0" "transformers>=4.46.0" "datasets>=2.20.0" "accelerate>=0.34.0" "safetensors>=0.4.5" psutil

In [ ]:
!rm -rf /content/ckpt_ae*; for n in 8 4 2; do echo "=== anchor-every $n ==="; python -u -m train.distill_colab --no-edge-hints --anchor-every $n --batch-size 4 --grad-accum 4 --steps 500 --stage1-steps 500 --save-every 250 --log-every 25 --keep-last 1 --ckpt-dir /content/ckpt_ae$n 2>&1 | tee /content/ablate_ae$n.log | grep --line-buffered -E "CKPT VERIFY|^\[step|Error|error|Traceback|Hardware"; done

In [ ]:
import json, pathlib
for n in (8, 4, 2):
    p = pathlib.Path(f"/content/ckpt_ae{n}/latest_verification.json")
    if p.is_file():
        v = json.loads(p.read_text())
        print(n, v["step"], "teacher", v["teacher_val_ppl"], "student", v["val_ppl_active"], "retention%", v["active_ppl_retention_pct"], "recall", v["val_lookahead_recall_at_k"])

## Delta-rule recurrence

Same 7:1 layout and settings as the first run, but recurrence layers erase a key's old value before writing (`--delta-rule`).

In [ ]:
!cd /content/terra_llm && git pull -q && python -u -m train.distill_colab --delta-rule --no-edge-hints --anchor-every 8 --batch-size 4 --grad-accum 4 --steps 500 --stage1-steps 500 --save-every 250 --log-every 25 --keep-last 1 --ckpt-dir /content/ckpt_delta8 2>&1 | tee /content/ablate_delta8.log | grep --line-buffered -E "CKPT VERIFY|^\[step|Error|error|Traceback|Hardware"
!python -c "import json; v=json.load(open('/content/ckpt_delta8/latest_verification.json')); print('delta8', v['step'], 'teacher', v['teacher_val_ppl'], 'student', v['val_ppl_active'], 'retention%', v['active_ppl_retention_pct'], 'recall', v['val_lookahead_recall_at_k'])"

## Write gate supervised by teacher attention

Delta rule plus a per-token write gate pulled toward the attention the teacher's later tokens pay that token (`--write-gate --write-gate-sup 0.25`). The step log shows mean gate / mean target as `wg=`.

In [ ]:
!cd /content/terra_llm && git pull -q && python -u -m train.distill_colab --delta-rule --write-gate --write-gate-sup 0.25 --no-edge-hints --anchor-every 8 --batch-size 4 --grad-accum 4 --steps 500 --stage1-steps 500 --save-every 250 --log-every 25 --keep-last 1 --ckpt-dir /content/ckpt_delta8_wgs 2>&1 | tee /content/ablate_delta8_wgs.log | grep --line-buffered -E "CKPT VERIFY|^\[step|Error|error|Traceback|Hardware"